In [6]:
import os
import sys
# Add the path to the src directory (two levels up)
sys.path.append(os.path.abspath('../../'))
from src.data_downloader import CFSDownloader
from src.data_processor import CFSProcessor
from src.database_utils import CFSDatabase

In [7]:
# Directory where the repository is cloned
local_path = '/Users/ljob/Desktop/'

# Path to data directory
input_dir = local_path + 'cnbs-predictor/data/'

# Path the GL mask file
mask_file = input_dir + 'input/GL_mask.nc'

# Path to save downloaded data
download_dir = local_path + 'Data/CFS/'

# Path to the CFS forecast database
database = local_path + 'Data/cfs_forecast_data.db'
table = 'cfs_forecast_data'

# Data source: specify either 'aws' or 'ncei'
source = 'aws'

# Do you need to download CFS data? ('yes' or 'no')
download_cfs = 'yes'

# Do you want to process the CFS data? ('yes' or 'no')
process_cfs = 'yes'

# Should grib files be deleted after processing? ('yes' or 'no')
delete_files = 'no'

# Auto mode will automatically open the existing database, pull the last entered date to determine the start date, 
# and set the end date to yesterday, making the database 'up-to-date'. If 'no', you can manually enter a start and 
# end date (ideal for testing or if you need to redownload/reprocess specific time frames).
auto = 'yes'

# Specify the start and end dates if auto mode above is set to 'no' (Format: MM-DD-YYYY)
start_date = '09-15-2025'
end_date = '09-16-2025'


In [18]:
import os
import urllib.request
import requests
import boto3
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
from botocore import UNSIGNED
from botocore.config import Config
from concurrent.futures import ThreadPoolExecutor
from io import StringIO
import pandas as pd

from src.utilities import check_url_exists

class CFSDownloader:
    def __init__(self):
        """
        Initialize the DataDownloader class.

        This method configures an S3 client for public access to download CFS data.
        """
        self.s3_config = Config(signature_version=UNSIGNED)
        self.s3 = boto3.client('s3', config=self.s3_config)


    def download_6hrly(self, download_dir, start_date, end_date=None, hours=None, products=None, source=None, bucket_name=None):
        """
        Download Climate Forecast System v2 (CFSv2) GRIB2 forecast files from AWS or NCEI over a specified date range.

        This method automates the download of monthly mean CFSv2 GRIB2 forecast files for specified hours, products,
        and source (AWS or NCEI). Files are organized by date and stored in local subdirectories named after each date.

        Parameters
        ----------
        download_dir : str
            Path to the local root directory where all downloaded files will be stored.

        start_date : str
            Start date for the download period, in 'MM-DD-YYYY' format.

        end_date : str, optional
            End date for the download period, in 'MM-DD-YYYY' format.
            If not provided, only start_date will be used.

        hours : list of str, optional
            List of forecast cycle hours to retrieve. Valid options are ['00', '06', '12', '18'].
            If not provided, all hours are downloaded.

        products : list of str, optional
            List of product prefixes to filter downloads. Typical values include:
            - 'pgbf': Pressure-level fields
            - 'flxf': Flux fields
            - 'ocnf': Ocean forecast fields
            - 'ipvf': Intermediate forecast variables
            If not provided, all default products are downloaded.

        source : str, optional
            The data source to use. Must be either:
            - 'aws': Download from NOAA AWS public dataset (default)
            - 'ncei': Download from NOAA's National Centers for Environmental Information

        bucket_name : str, optional
            The name of the AWS S3 bucket. Defaults to 'noaa-cfs-pds' when using AWS.
            Not used when `source='ncei'`.

        Raises
        ------
        ValueError
            If date formats are incorrect, if invalid product names are provided, or if an invalid source is specified.

        Notes
        -----
        - If only start_date is provided, downloads only that day.
        - Otherwise, downloads all dates from start_date (inclusive) to end_date (exclusive).
        - Skips missing files or directories gracefully.
        - AWS downloads use S3 API calls; NCEI uses HTML scraping.
        """

        os.makedirs(download_dir, exist_ok=True)

        # Default values
        if hours is None:
            hours = ['00', '06', '12', '18']
        if products is None:
            products = ['pgbf', 'flxf', 'ocnf', 'ipvf']
        if source is None:
            source = 'aws'
        if bucket_name is None:
            bucket_name = 'noaa-cfs-pds'

        # Validate and parse start_date
        try:
            start = datetime.strptime(start_date, "%m-%d-%Y")
        except ValueError:
            raise ValueError("ERROR: start_date must be in 'MM-DD-YYYY' format.")

        # Handle optional end_date
        if end_date:
            try:
                end = datetime.strptime(end_date, "%m-%d-%Y")
            except ValueError:
                raise ValueError("ERROR: end_date must be in 'MM-DD-YYYY' format.")
            if end < start:
                raise ValueError("ERROR: end_date must be after or equal to start_date")
        else:
            end = start + timedelta(days=1)

        # Validate products
        if not isinstance(products, list) or not all(isinstance(p, str) for p in products):
            raise ValueError("ERROR: Products must be a list of strings.")
        valid_products = {'pgbf', 'flxf', 'ocnf', 'ipvf', 'pgb', 'flx', 'ocn', 'ipv'}

        delta = timedelta(days=1)
        current = start

        while current < end:
            YYYY = current.strftime("%Y")
            MM = current.strftime("%m")
            DD = current.strftime("%d")
            date_folder = current.strftime("%Y%m%d")
            date_dir = os.path.join(download_dir, date_folder)
            os.makedirs(date_dir, exist_ok=True)

            for HH in hours:
                for product in products:
                    if product not in valid_products:
                        print(f"ERROR: No files matching '{product}'. Valid products are: {', '.join(valid_products)}. Skipping.")
                        continue

                    if source.lower() == 'aws':
                        url_path = f'cfs.{YYYY}{MM}{DD}/{HH}/6hrly_grib_01/'

                        # Check if any files exist in AWS
                        continuation_token = None
                        objects_found = False
                        while True:
                            list_args = {'Bucket': bucket_name, 'Prefix': url_path}
                            if continuation_token:
                                list_args['ContinuationToken'] = continuation_token

                            response = self.s3.list_objects_v2(**list_args)
                            contents = response.get('Contents', [])
                            if any(product in os.path.basename(obj['Key']) and obj['Key'].endswith('.grb2') for obj in contents):
                                objects_found = True
                                break
                            if not response.get('IsTruncated', False):
                                break
                            continuation_token = response.get('NextContinuationToken')

                        if not objects_found:
                            print(f"No AWS files for '{product}' on {MM}-{DD}-{YYYY} {HH}. Skipping.")
                            continue

                        print(url_path)
                        self._from_aws(product, bucket_name, url_path, date_dir)

                    elif source.lower() == 'ncei':
                        base_url = 'https://www.ncei.noaa.gov/data/climate-forecast-system/access/operational-9-month-forecast/monthly-means'
                        url_path = f'{base_url}/{YYYY}/{YYYY}{MM}/{YYYY}{MM}{DD}/{YYYY}{MM}{DD}{HH}/'

                        if not check_url_exists(url_path):
                            print(f"No NCEI URL for {product} on {MM}-{DD}-{YYYY} {HH}. Skipping.")
                            continue

                        try:
                            response = urllib.request.urlopen(url_path)
                            soup = BeautifulSoup(response.read().decode('utf-8'), 'html.parser')
                            links = [link for link in soup.find_all('a') if link['href'].startswith(product) and link['href'].endswith('.grb2')]
                            if not links:
                                print(f"No NCEI files for '{product}' on {MM}-{DD}-{YYYY} {HH}. Skipping.")
                                continue
                        except Exception as e:
                            print(f"Error accessing NCEI URL for {MM}-{DD}-{YYYY} {HH}: {e}")
                            continue

                        self._from_ncei(product, url_path, date_dir)

                    else:
                        raise ValueError('Invalid source. Source must be either "aws" or "ncei".')

            current += delta

        print("Download completed successfully.")


    def _from_aws(self, product, bucket_name, url_path, download_dir):
        continuation_token = None
        objects = []

        while True:
            list_args = {'Bucket': bucket_name, 'Prefix': url_path}
            if continuation_token:
                list_args['ContinuationToken'] = continuation_token

            response = self.s3.list_objects_v2(**list_args)
            contents = response.get('Contents', [])
            if not contents:
                print(f"No files found in AWS path: {url_path}")
                break

            objects.extend(contents)
            if not response.get('IsTruncated', False):
                break
            continuation_token = response.get('NextContinuationToken')

        # Filter keys for products
        keys_to_download = [obj['Key'] for obj in objects if product in os.path.basename(obj['Key']) and obj['Key'].endswith('grib.grb2')]

        def download_key(key):
            filename = os.path.basename(key)
            local_path = os.path.join(download_dir, filename)
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            self.s3.download_file(bucket_name, key, local_path)
            print(f"Downloaded from AWS: {filename}")

        with ThreadPoolExecutor(max_workers=5) as executor:
            executor.map(download_key, keys_to_download)

In [ ]:
https://noaa-cfs-pds.s3.amazonaws.com/cfs.20181208/00/6hrly_grib_01/pgbf2018121200.01.2018120800.grb2
start_date = '12-08-2018'

cfs_downloader = CFSDownloader()

cfs_downloader.download_6hrly(
    download_dir, 
    start_date,
    end_date=None,
    hours=['00'],
    products=['pgbf'],
    source='aws'
    )

cfs.20181208/00/6hrly_grib_01/
Download completed successfully.


In [22]:
import requests
import os
import time
from datetime import datetime, timedelta

# Base URL
BASE_URL = "https://noaa-cfs-pds.s3.amazonaws.com/cfs.20240101/00/6hrly_grib_01/"

# Output directory
DOWNLOAD_DIR = "/Users/ljob/Desktop/Data/CFS/6hrly/20240101/"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Start datetime
start_date = datetime(2024, 1, 1, 0)

# End date = start + 9 months (~270 days)
end_date = start_date + timedelta(days=270)

# Fixed part of filename
FIXED_PART = "01.2024010100.grb2"

# Timing
total_start = time.time()
total_download_time = 0
file_count = 0

current_date = start_date

while current_date <= end_date:

    # Format date
    date_str = current_date.strftime("%Y%m%d%H")

    # Build filename
    filename = f"pgbf{date_str}.{FIXED_PART}"

    # Full URL
    file_url = BASE_URL + filename

    # Local path
    output_path = os.path.join(DOWNLOAD_DIR, filename)

    print(f"Downloading: {filename}")

    start_time = time.time()

    try:
        response = requests.get(file_url, stream=True, timeout=60)

        if response.status_code == 200:

            with open(output_path, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

            elapsed = time.time() - start_time
            total_download_time += elapsed
            file_count += 1

            print(f"  Done in {elapsed:.2f} seconds")

        else:
            print(f"  Skipped (HTTP {response.status_code})")

    except Exception as e:
        print(f"  Error: {e}")

    # Increment by 6 hours
    current_date += timedelta(hours=6)


total_elapsed = time.time() - total_start

print("\n" + "=" * 50)
print("Download Complete")
print(f"Files downloaded: {file_count}")
print(f"Total download time (sum): {total_download_time:.2f} sec")
print(f"Total wall time: {total_elapsed:.2f} sec")
print("=" * 50)


Downloading: pgbf2024010100.01.2024010100.grb2
  Done in 1.29 seconds
Downloading: pgbf2024010106.01.2024010100.grb2
  Done in 1.19 seconds
Downloading: pgbf2024010112.01.2024010100.grb2
  Done in 1.07 seconds
Downloading: pgbf2024010118.01.2024010100.grb2
  Done in 1.33 seconds
Downloading: pgbf2024010200.01.2024010100.grb2
  Done in 1.59 seconds
Downloading: pgbf2024010206.01.2024010100.grb2
  Done in 1.21 seconds
Downloading: pgbf2024010212.01.2024010100.grb2
  Done in 1.13 seconds
Downloading: pgbf2024010218.01.2024010100.grb2
  Done in 2.32 seconds
Downloading: pgbf2024010300.01.2024010100.grb2
  Done in 1.16 seconds
Downloading: pgbf2024010306.01.2024010100.grb2
  Done in 1.08 seconds
Downloading: pgbf2024010312.01.2024010100.grb2
  Done in 1.25 seconds
Downloading: pgbf2024010318.01.2024010100.grb2
  Done in 1.29 seconds
Downloading: pgbf2024010400.01.2024010100.grb2
  Done in 1.18 seconds
Downloading: pgbf2024010406.01.2024010100.grb2
  Done in 1.18 seconds
Downloading: pgbf202